# 실습 전 결측 처리
- 1주차 1-4에서 정의했던 것과 같이 결측처리 진행
- 48시간 이상을 연속 결측으로 정의
- 연속 결측은 마스킹 제거, 산발적 결측은 선형 보간으로 결측 처리
- 구조적 결측은 없음

In [3]:
import os
import numpy as np
import pandas as pd

CONTINUOUS_THRESHOLD = 48  

def stuck_mask(values, min_len=2):
    n = len(values)
    mask = np.zeros(n, dtype=bool)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and values[j + 1] == values[i]:
            j += 1
        if j - i + 1 >= min_len:
            mask[i:j + 1] = True
        i = j + 1
    return mask

subset = pd.read_csv("../Data/raw/subset_17_buildings.csv", encoding="utf-8")
missing_profile = pd.read_csv("../results/wk1/wk01_missing_profile.csv", encoding="utf-8")
type_map = missing_profile.set_index("building_id")["유형"]

RAW_DIR = "../Data/raw/bdg2"
electricity = pd.read_csv(f"{RAW_DIR}/meters/electricity.csv", encoding="utf-8")
electricity["timestamp"] = pd.to_datetime(electricity["timestamp"])
electricity = electricity.set_index("timestamp")

building_ids = subset["building_id"].tolist()

treated = {}
for bid in building_ids:
    s = electricity[bid]
    values = s.fillna(-999999).values
    mask = stuck_mask(values, min_len=2) | s.isna().values  # 고정값+NaN 병합 결측 지점

    col = s.copy()
    col[mask] = np.nan
    if type_map[bid] != "연속":
        col = col.interpolate(method="linear")  # 산발적: 선형보간(A)
    # 연속: 마스킹 제거 -> NaN으로 남김(C), 채우지 않음

    treated[bid] = col

treated_df = pd.DataFrame(treated)

os.makedirs("../Data/processed", exist_ok=True)
out_path = "../Data/processed/subset_17_buildings_Impulation.csv"
treated_df.to_csv(out_path, encoding="utf-8")

print("처리 방식별 건물 수:")
print(type_map.loc[building_ids].value_counts())
print(f"\n저장 완료: {out_path}")
print("잔여 결측치(연속 유형에서 마스킹으로 남긴 부분):", int(treated_df.isna().sum().sum()))


처리 방식별 건물 수:
유형
산발적    15
연속      2
Name: count, dtype: int64

저장 완료: ../Data/processed/subset_17_buildings_Impulation.csv
잔여 결측치(연속 유형에서 마스킹으로 남긴 부분): 337


In [4]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.backtest import rolling_backtest

data = pd.read_csv(
    "../Data/processed/subset_17_buildings_Impulation.csv",
    encoding="utf-8", index_col=0, parse_dates=True,
)

print("데이터 shape:", data.shape, "| 기간:", data.index.min(), "~", data.index.max())
print("잔여 결측치(연속 유형 마스킹 지점):", int(data.isna().sum().sum()))


데이터 shape: (17544, 17) | 기간: 2016-01-01 00:00:00 ~ 2017-12-31 23:00:00
잔여 결측치(연속 유형 마스킹 지점): 337


# 과제 2-1. 백테스트 하네스 구현

In [5]:
SEASON = 168  # Seasonal Naive 주기(1주=168시간) — 하네스 검증용 임시 모델

def seasonal_naive_model_fn(train_df):
    return train_df  # 학습이 필요 없으므로 train_df 자체를 "모델"로 사용

def seasonal_naive_predict_fn(model, horizon, season=SEASON):
    tail = model.iloc[-season:]           # train의 마지막 1주
    return tail.iloc[:horizon].reset_index(drop=True)  # y(t-168)을 그대로 예측값으로 사용

# step_size=168(1주일)로 origin을 horizon과 분리 -> window가 전체 기간(2017년)에 고르게 퍼짐
STEP_SIZE = 168
N_WINDOWS = 52  # 1년 학습창 + 1주 이동 시 나오는 최대 개수(옵션 A)

# 연구 질문 1: expanding vs sliding, horizon 24h vs 168h
configs = [
    ("sliding", 24),
    ("expanding", 24),
    ("sliding", 168),
    ("expanding", 168),
]

summary_rows = []
for train_mode, horizon in configs:
    res = rolling_backtest(
        data, seasonal_naive_model_fn, seasonal_naive_predict_fn,
        n_windows=N_WINDOWS, horizon=horizon, train_mode=train_mode,
        sliding_train_len=8760, step_size=STEP_SIZE,
    )

    mae = (res["y_true"] - res["y_pred"]).abs().mean()

    summary_rows.append({
        "train_mode": train_mode,
        "horizon": horizon,
        "테스트 구간": f"{res['timestamp'].min().date()} ~ {res['timestamp'].max().date()}",
        "MAE(Seasonal Naive)": round(mae, 4),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


train_mode  horizon                  테스트 구간  MAE(Seasonal Naive)
   sliding       24 2017-01-08 ~ 2017-12-31              29.9028
 expanding       24 2017-01-08 ~ 2017-12-31              29.9028
   sliding      168 2017-01-02 ~ 2017-12-31              26.8567
 expanding      168 2017-01-02 ~ 2017-12-31              26.8567


In [6]:
# 연구 질문 2: window(origin) 개수를 늘릴 때의 변화 (train_mode=expanding, horizon=24h, step_size=168h 고정)
window_counts = [8, 12, 16, 20, 32, 52]

wc_rows = []
for nw in window_counts:
    res = rolling_backtest(
        data, seasonal_naive_model_fn, seasonal_naive_predict_fn,
        n_windows=nw, horizon=24, train_mode="expanding", step_size=STEP_SIZE,
    )

    mae = (res["y_true"] - res["y_pred"]).abs().mean()
    per_window_mae = res.groupby("window").apply(
        lambda g: (g["y_true"] - g["y_pred"]).abs().mean(), include_groups=False
    )

    wc_rows.append({
        "n_windows": nw,
        "테스트 구간": f"{res['timestamp'].min().date()} ~ {res['timestamp'].max().date()}",
        "전체 MAE": round(mae, 4),
        "window별 MAE 표준편차": round(per_window_mae.std(), 4),
    })

wc_df = pd.DataFrame(wc_rows)
print(wc_df.to_string(index=False))


 n_windows                  테스트 구간  전체 MAE  window별 MAE 표준편차
         8 2017-11-12 ~ 2017-12-31 59.5677           82.0588
        12 2017-10-15 ~ 2017-12-31 44.6989           69.1004
        16 2017-09-17 ~ 2017-12-31 40.7742           59.8499
        20 2017-08-20 ~ 2017-12-31 36.4077           53.9375
        32 2017-05-28 ~ 2017-12-31 38.0095           45.5681
        52 2017-01-08 ~ 2017-12-31 29.9028           37.2444


# 과제 2-2. 지표 모듈 구현

In [7]:
from src.metrics import compute_naive_denominators, score_backtest

res_demo = pd.read_csv("../results/wk2/backtest_expanding_24h_w52_step168.csv", encoding="utf-8")
res_demo["timestamp"] = pd.to_datetime(res_demo["timestamp"])

first_test_idx = data.index.get_loc(res_demo["timestamp"].min())
naive_denominators = compute_naive_denominators(data, train_end=first_test_idx, season=168)

per_building, overall = score_backtest(res_demo, naive_denominators)

print("건물별 지표 (상위 5개):")
print(per_building.sort_values("MASE").head().to_string(index=False))
print("\n전체 집계:")
for k, v in overall.items():
    print(f"  {k}: {v:.4f}")


건물별 지표 (상위 5개):
            building       MAE      RMSE     MASE      wQL
    Rat_office_Randy  5.479135 13.250109 0.428699 0.046750
Bear_education_Lewis  1.455211  1.885817 0.521798 0.043302
Rat_assembly_Pauline  0.488673  0.955623 0.646205 0.194957
Fox_education_Elvira 14.854736 24.247944 0.654199 0.048792
 Rat_public_Julieann  0.741196  2.093660 0.678667 0.081849

전체 집계:
  MAE_단순평균: 29.8913
  RMSE_단순평균: 72.6414
  MASE_평균: 0.9673
  wQL_평균: 0.1244


# 과제 2-3. Seasonal Naive 베이스라인 확정

In [8]:
#최종 설계 결정: expanding + 24h or 168h + 52 windows
#step_size = 168 (1주)

FINAL_TRAIN_MODE = "expanding"

baseline_rows = []
per_building_all = []
for horizon in [24, 168]:
    res = rolling_backtest(
        data, seasonal_naive_model_fn, seasonal_naive_predict_fn,
        n_windows=N_WINDOWS, horizon=horizon, train_mode=FINAL_TRAIN_MODE,
        step_size=STEP_SIZE,
    )
    res.to_csv(
        f"../results/wk2/backtest_{FINAL_TRAIN_MODE}_{horizon}h_w{N_WINDOWS}_step{STEP_SIZE}.csv",
        index=False, encoding="utf-8",
    )

    first_test_idx = data.index.get_loc(res["timestamp"].min())
    denom = compute_naive_denominators(data, train_end=first_test_idx, season=168)
    per_building, overall = score_backtest(res, denom)
    per_building["horizon"] = horizon
    per_building_all.append(per_building)

    baseline_rows.append({
        "horizon": horizon,
        "train_mode": FINAL_TRAIN_MODE,
        "n_windows": N_WINDOWS,
        "step_size": STEP_SIZE,
        **overall,
    })
    print(f"horizon={horizon}h  MASE_평균={overall['MASE_평균']:.4f}  (1.0에 가까우면 정상 -- 분모 정의 검산)")

baseline_summary = pd.DataFrame(baseline_rows)
baseline_per_building = pd.concat(per_building_all, ignore_index=True)

os.makedirs("../results", exist_ok=True)
baseline_summary.to_csv("../results/baseline_seasonal_naive.csv", index=False, encoding="utf-8")
baseline_per_building.to_csv("../results/wk2/baseline_seasonal_naive_per_building.csv", index=False, encoding="utf-8")

print()
print(baseline_summary.to_string(index=False))
print("\n저장 완료: ../results/baseline_seasonal_naive.csv")


horizon=24h  MASE_평균=0.9673  (1.0에 가까우면 정상 -- 분모 정의 검산)
horizon=168h  MASE_평균=0.9767  (1.0에 가까우면 정상 -- 분모 정의 검산)

 horizon train_mode  n_windows  step_size  MAE_단순평균  RMSE_단순평균  MASE_평균   wQL_평균
      24  expanding         52        168 29.891289  72.641364 0.967272 0.124353
     168  expanding         52        168 26.855319  61.138230 0.976729 0.100042

저장 완료: ../results/baseline_seasonal_naive.csv


# 과제 2-4. 누수 검증

In [9]:
from src.metrics import _naive_train_mae

res_leak = rolling_backtest(
    data, seasonal_naive_model_fn, seasonal_naive_predict_fn,
    n_windows=N_WINDOWS, horizon=24, train_mode=FINAL_TRAIN_MODE, step_size=STEP_SIZE,
)
first_test_idx = data.index.get_loc(res_leak["timestamp"].min())

# 정상: 테스트 시작 이전 데이터로만 분모 계산 
denom_correct = compute_naive_denominators(data, train_end=first_test_idx, season=168)

# 누수: 테스트 구간(2017-01-08~12-31) 자체로 분모 계산 -> 평가 대상 기간의 실제값이 분모에 들어감
test_ref = data.iloc[first_test_idx:]
denom_leaky = {bid: _naive_train_mae(test_ref[bid], season=168) for bid in data.columns}

_, overall_correct = score_backtest(res_leak, denom_correct)
_, overall_leaky = score_backtest(res_leak, denom_leaky)

mase_correct = overall_correct["MASE_평균"]
mase_leaky = overall_leaky["MASE_평균"]
diff_pct = (mase_correct - mase_leaky) / mase_correct * 100

print(f"정상(학습 구간 분모)  MASE_평균 = {mase_correct:.4f}")
print(f"누수(테스트 구간 분모) MASE_평균 = {mase_leaky:.4f}")
print(f"누수 버전이 {diff_pct:.1f}% {'더 좋게(작게)' if diff_pct > 0 else '더 나쁘게'} 나옴")

leak_result = pd.DataFrame([
    {"버전": "정상(학습 구간 분모)", "MASE_평균": mase_correct},
    {"버전": "누수(테스트 구간 분모)", "MASE_평균": mase_leaky},
])
leak_result.to_csv("../results/wk2/wk02_leakage_experiment.csv", index=False, encoding="utf-8")
print("\n저장 완료: ../results/wk2/wk02_leakage_experiment.csv")


정상(학습 구간 분모)  MASE_평균 = 0.9673
누수(테스트 구간 분모) MASE_평균 = 1.0186
누수 버전이 -5.3% 더 나쁘게 나옴

저장 완료: ../results/wk2/wk02_leakage_experiment.csv
